# 歷史版本 C：Update 05/23 – 分群

**用途**：05/23 版本的 K-means、階層式（Pearson／Spearman 相關距離）與 Scipy vs Sklearn 對照。

| 項目 | 說明 |
|---|---|
| 輸入 | data/raw CSV |
| 輸出 | （歷史探索，圖表為即時輸出） |
| 來源 | MVA 期末報告分析.ipynb（Colab 原檔，191 cells） 之 cell 74–95 |

> 歷史版本，保留供追溯；不保證可獨立執行，最終流程請見 ../final_analysis/ 與 ../../src/。


In [ ]:
# 共用設定：載入資料並提供 df / pivot_table_age_std / pivot_table_cleaned / scaled_data / taiwan_map / prop 等變數
# （取代原 Colab 的 Drive 掛載、字型下載與 GDAL 設定；詳見 _legacy_setup.py）
import sys; sys.path.insert(0, '..')   # _legacy_setup.py 位於 notebooks/
from _legacy_setup import *
df = pd.read_csv(CSV_PATH)  # 早期版本使用原始中文欄位名稱（覆蓋共用設定的英文欄位版）

## Cluster Analysis

We will now perform cluster analysis on the `pivot_table_age_std` to group similar `縣市別` (cities/counties) based on their age-standardized incidence rates over the years. We'll test different distance measures and clustering algorithms as requested.

### K-Means Clustering

We'll apply K-Means clustering. For demonstration, we'll choose a number of clusters (e.g., `k=4`). To visualize the clusters, we'll reduce the dimensionality using PCA to two components.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# 1. Perform K-means with k=4
k = 4
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(scaled_data)

# 2. Map numeric labels to A, B, C, D
cluster_map = {0: 'Group A', 1: 'Group B', 2: 'Group C', 3: 'Group D'}
encoded_labels = [cluster_map[label] for label in kmeans_labels]

# 3. Create plotting DataFrame
counties_names = pivot_table_cleaned.index
pca_kmeans = PCA(n_components=2)
components_kmeans = pca_kmeans.fit_transform(scaled_data)

df_kmeans_plot = pd.DataFrame({
    'PC1': components_kmeans[:, 0],
    'PC2': components_kmeans[:, 1],
    'County': counties_names,
    'Cluster': encoded_labels
})

# 4. Visualization with high-contrast palette
plt.figure(figsize=(12, 8))
sns.scatterplot(
    data=df_kmeans_plot,
    x='PC1',
    y='PC2',
    hue='Cluster',
    hue_order=['Group A', 'Group B', 'Group C', 'Group D'],
    palette='bright',
    s=200,
    alpha=0.9,
    edgecolor='black'
)

# Annotate county names
for i in range(len(df_kmeans_plot)):
    plt.annotate(
        df_kmeans_plot['County'].iloc[i],
        (df_kmeans_plot['PC1'].iloc[i], df_kmeans_plot['PC2'].iloc[i]),
        textcoords='offset points',
        xytext=(0, 12),
        ha='center',
        fontsize=11,
        fontname=prop.get_name() if 'prop' in locals() else None,
        bbox=dict(boxstyle='round,pad=0.2', fc='yellow', alpha=0.3)
    )

plt.title(f'K-Means Clustering (k={k}) - High Contrast View', fontsize=16)
plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)')
plt.grid(True, alpha=0.3)
plt.axhline(0, color='black', linestyle='--', linewidth=0.8)
plt.axvline(0, color='black', linestyle='--', linewidth=0.8)
plt.legend(title='Clusters', loc='best')
plt.show()

# 5. Print the list of counties per cluster
print("\n--- Cluster Assignments ---")
for group in ['Group A', 'Group B', 'Group C', 'Group D']:
    members = df_kmeans_plot[df_kmeans_plot['Cluster'] == group]['County'].tolist()
    print(f"{group}: {', '.join(members)}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# 1. Perform K-means with k=2
k = 2
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(scaled_data)

# 2. Map numeric labels to Group A, B
cluster_map = {0: 'Group A', 1: 'Group B'}
encoded_labels = [cluster_map[label] for label in kmeans_labels]

# 3. Create plotting DataFrame
counties_names = pivot_table_cleaned.index
pca_kmeans = PCA(n_components=2)
components_kmeans = pca_kmeans.fit_transform(scaled_data)

df_kmeans_plot = pd.DataFrame({
    'PC1': components_kmeans[:, 0],
    'PC2': components_kmeans[:, 1],
    'County': counties_names,
    'Cluster': encoded_labels
})

# 4. Visualization with high-contrast palette
plt.figure(figsize=(12, 8))
sns.scatterplot(
    data=df_kmeans_plot,
    x='PC1',
    y='PC2',
    hue='Cluster',
    hue_order=['Group A', 'Group B'],
    palette='bright',
    s=200,
    alpha=0.9,
    edgecolor='black'
)

# Annotate county names
for i in range(len(df_kmeans_plot)):
    plt.annotate(
        df_kmeans_plot['County'].iloc[i],
        (df_kmeans_plot['PC1'].iloc[i], df_kmeans_plot['PC2'].iloc[i]),
        textcoords='offset points',
        xytext=(0, 12),
        ha='center',
        fontsize=11,
        fontname=prop.get_name() if 'prop' in locals() else None,
        bbox=dict(boxstyle='round,pad=0.2', fc='yellow', alpha=0.3)
    )

plt.title(f'K-Means Clustering (k={k}) - High Contrast View', fontsize=16)
plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)')
plt.grid(True, alpha=0.3)
plt.axhline(0, color='black', linestyle='--', linewidth=0.8)
plt.axvline(0, color='black', linestyle='--', linewidth=0.8)
plt.legend(title='Clusters', loc='best')
plt.show()

# 5. Print the list of counties per cluster
print(f"\n--- Cluster Assignments (k={k}) ---")
for group in ['Group A', 'Group B']:
    members = df_kmeans_plot[df_kmeans_plot['Cluster'] == group]['County'].tolist()
    print(f"{group}: {', '.join(members)}")

In [ ]:
# 1. Prepare Group Data for both k=2 and k=4
# K=2 Grouping
k2_group_a = df_kmeans_plot[df_kmeans_plot['Cluster'] == 'Group A']['County'].tolist()
k2_group_b = df_kmeans_plot[df_kmeans_plot['Cluster'] == 'Group B']['County'].tolist()

# K=4 Grouping (Calculated dynamically)
k4_kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
k4_labels = k4_kmeans.fit_predict(scaled_data)
cluster_map_4 = {0: 'Group A', 1: 'Group B', 2: 'Group C', 3: 'Group D'}
df_k4 = pd.DataFrame({'County': pivot_table_cleaned.index, 'Cluster': [cluster_map_4[l] for l in k4_labels]})

# 2. Process Data for Plotting
df_long_k2 = df_long.copy()
df_long_k2['Macro_Group'] = df_long_k2['City_County'].apply(lambda x: 'Group A' if x in k2_group_a else 'Group B')
df_group_means_k2 = df_long_k2.groupby(['Year', 'Macro_Group'])['Incidence_Rate'].mean().reset_index()

df_long_k4 = df_long.merge(df_k4, left_on='City_County', right_on='County', suffixes=('', '_k4'))
df_group_means_k4 = df_long_k4.groupby(['Year', 'Cluster_k4'])['Incidence_Rate'].mean().reset_index()

# 3. Create 2x1 Subplot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 12), sharex=True)

# Top Plot: k=2
sns.lineplot(data=df_group_means_k2, x='Year', y='Incidence_Rate', hue='Macro_Group',
             palette=['#6EA1FF', '#FFB366'], linewidth=3, ax=ax1)
ax1.set_title('Annual Average Incidence: Macro Groups (k=2)', fontsize=14)
ax1.set_ylabel('Mean Rate (per 100k)')
ax1.grid(True, alpha=0.3)

# Bottom Plot: k=4
sns.lineplot(data=df_group_means_k4, x='Year', y='Incidence_Rate', hue='Cluster_k4',
             palette='bright', linewidth=3, ax=ax2)
ax2.set_title('Annual Average Incidence: Detailed Groups (k=4)', fontsize=14)
ax2.set_ylabel('Mean Rate (per 100k)')
ax2.set_xlabel('Diagnosis Year')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 4. Summary Stats
print("--- Growth Summary (1979-2022) ---")
for group in df_group_means_k2['Macro_Group'].unique():
    subset = df_group_means_k2[df_group_means_k2['Macro_Group'] == group]
    start, end = subset.iloc[0]['Incidence_Rate'], subset.iloc[-1]['Incidence_Rate']
    print(f"{group} (k=2): {start:.2f} -> {end:.2f} ({((end-start)/start)*100:.1f}%)")

### Hierarchical Clustering

Now, let's explore hierarchical clustering using different distance metrics and linkage methods. We'll use Pearson Correlation and Spearman Correlation as distance measures, and 'single', 'complete', and 'average' as linkage methods. For each combination, we'll display a dendrogram.

#### Hierarchical Clustering with Pearson Correlation Distance

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
import matplotlib.pyplot as plt
import pandas as pd

# 設定分群數
k_clusters = 4
linkage_methods = ['single', 'complete', 'average']
counties = clustering_data_clean.index.tolist()

plt.figure(figsize=(18, 15))

for i, method in enumerate(linkage_methods):
    # 1. 執行階層分群
    linked = linkage(pearson_distances, method=method)

    # 2. 繪製樹狀圖
    plt.subplot(3, 1, i + 1)
    dendrogram(linked,
               orientation='top',
               labels=counties,
               distance_sort='descending',
               show_leaf_counts=True,
               color_threshold=0.7 * max(linked[:,2]))
    plt.title(f'Hierarchical Clustering (Pearson - {method.capitalize()} Linkage)')
    plt.xlabel('County/City')
    plt.ylabel('Distance')

    # 3. 取得分群結果並列印
    cluster_labels = fcluster(linked, k_clusters, criterion='maxclust')
    df_clusters = pd.DataFrame({'County': counties, 'Cluster': cluster_labels})

    print(f"\n--- {method.capitalize()} Linkage Cluster Results (k={k_clusters}) ---")
    for c in range(1, k_clusters + 1):
        members = df_clusters[df_clusters['Cluster'] == c]['County'].tolist()
        print(f"Group {c}: {', '.join(members)}")

plt.tight_layout()
plt.show()

In [ ]:
from scipy.cluster.hierarchy import linkage
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
import pandas as pd

# 1. 定義參數
methods = ['single', 'complete', 'average', 'ward']
k_fixed = 4
sil_results = []

# 2. 迴圈計算各個 linkage method 的 silhouette score
for method in methods:
    # 使用 AgglomerativeClustering 執行分群
    clusterer = AgglomerativeClustering(n_clusters=k_fixed, linkage=method)
    labels = clusterer.fit_predict(scaled_clustering_data)

    # 計算輪廓分數
    score = silhouette_score(scaled_clustering_data, labels)
    sil_results.append({
        'Linkage Method': method.capitalize(),
        'Clusters (k)': k_fixed,
        'Silhouette Score': round(score, 4)
    })

# 3. 以 DataFrame 輸出結果
df_sil_comparison = pd.DataFrame(sil_results)
print("--- Silhouette Score Comparison (k=4) ---")
display(df_sil_comparison)

# 4. 保留原有的 Elbow 趨勢分析 (以表現較佳的 Ward 為例)
K_range = range(2, 10)
sil_scores_trend = []

for k in K_range:
    clusterer = AgglomerativeClustering(n_clusters=k, linkage='ward')
    labels = clusterer.fit_predict(scaled_clustering_data)
    sil_scores_trend.append(silhouette_score(scaled_clustering_data, labels))

# 繪製趨勢圖
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 5))
plt.plot(K_range, sil_scores_trend, marker='o', color='tab:blue', label='Ward Silhouette Score')
plt.title('Silhouette Score Trend (Ward Linkage)')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Score')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 初始化參數
methods = ['single', 'complete', 'average', 'ward']
k_range = range(2, 11)
results = []

# 2. 巢狀迴圈計算分數
for k in k_range:
    for method in methods:
        clusterer = AgglomerativeClustering(n_clusters=k, linkage=method)
        labels = clusterer.fit_predict(scaled_clustering_data)
        score = silhouette_score(scaled_clustering_data, labels)
        results.append({
            'k': k,
            'Method': method.capitalize(),
            'Silhouette Score': score
        })

# 3. 轉換為 DataFrame
df_comparison = pd.DataFrame(results)

# 4. 建立樞紐表以便觀察
df_pivot = df_comparison.pivot(index='k', columns='Method', values='Silhouette Score')
print("--- Silhouette Score Comparison (k=2 to 10) ---")
display(df_pivot)

# 5. 繪製趨勢圖
plt.figure(figsize=(12, 6))
sns.lineplot(data=df_comparison, x='k', y='Silhouette Score', hue='Method', marker='o')
plt.title('Silhouette Score Trends by Linkage Methods', fontsize=14)
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.grid(True, linestyle='--', alpha=0.6)
plt.xticks(list(k_range))
plt.legend(title='Linkage Method')
plt.show()

In [ ]:
print(df_pivot)

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster
import matplotlib.pyplot as plt
import geopandas as gpd
from matplotlib.lines import Line2D
import pandas as pd

# 1. Prepare clustering setup
methods = ['single', 'complete', 'average', 'ward']
# k_values = [4,4,4,5] VS [2,2,2,2]
k_values = [4,4,4,5] # Number of clusters for each method
counties_cleaned = pivot_table_cleaned.index

# 2. Setup figure for 2x2 comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 16), dpi=100)
axes = axes.flatten()

# Define consistent color mapping (Extended to 5 for Ward linkage)
color_map = {
    1: '#00FFFF', # Cyan
    2: '#FFFF00', # Yellow
    3: '#FF00FF', # Magenta
    4: '#00FF00', # Lime
    5: '#FF8000'  # Orange
}

for i, method in enumerate(methods):
    current_k = k_values[i]
    # Perform linkage
    Z = linkage(pearson_distances, method=method)
    # Form flat clusters
    labels = fcluster(Z, current_k, criterion='maxclust')

    # Map labels to the counties
    df_temp = pd.DataFrame({'County': counties_cleaned, 'Cluster': labels})
    df_temp['County_Match'] = df_temp['County'].str.replace('台', '臺')

    # Merge with the map (taiwan_map already in WGS84)
    merged_temp = taiwan_map.merge(df_temp, left_on='COUNTYNAME', right_on='County_Match', how='left')
    merged_temp['plot_color'] = merged_temp['Cluster'].map(color_map).fillna('#e0e0e0')

    # Plot on the specific subplot
    ax = axes[i]
    merged_temp.plot(
        ax=ax,
        color=merged_temp['plot_color'],
        edgecolor='gray',
        linewidth=0.5
    )

    # Formatting
    ax.set_xlim(118.0, 122.5)
    ax.set_ylim(21.5, 26.5)
    ax.set_title(f'Linkage: {method.capitalize()} (k={current_k})', fontsize=14, fontweight='bold')
    ax.set_axis_off()

    # Add a simple text legend for cluster IDs
    legend_labels = [f"Group {idx}" for idx in range(1, current_k + 1)]
    custom_lines = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor=color_map[idx], markersize=10)
        for idx in range(1, current_k + 1)
    ]
    ax.legend(custom_lines, legend_labels, loc='upper left', fontsize=9, title='Clusters')

plt.suptitle('Comparison of Hierarchical Clustering Linkage Methods (Spatial)', fontsize=18, y=0.98)
plt.tight_layout()
plt.show()

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster
import pandas as pd

# 1. 準備資料與演算法
methods = ['single', 'complete', 'average', 'ward']
k_sets = {
    "Scenario 1 (k=[4,4,4,5])": [4, 4, 4, 5],
    "Scenario 2 (k=[2,2,2,2])": [2, 2, 2, 2]
}
counties = pivot_table_cleaned.index.tolist()

# 2. 執行並列印結果
for scenario_name, k_values in k_sets.items():
    print(f"\n=== {scenario_name} ===")
    results = {}

    for i, method in enumerate(methods):
        current_k = k_values[i]
        # 執行連線演算法
        Z = linkage(pearson_distances, method=method)
        # 取得分群標籤
        labels = fcluster(Z, current_k, criterion='maxclust')

        # 整理成群組清單
        cluster_groups = {}
        for county, cluster_id in zip(counties, labels):
            cluster_groups.setdefault(cluster_id, []).append(county)

        print(f"\nLinkage Method: {method.capitalize()} (k={current_k})")
        for cluster_id in sorted(cluster_groups.keys()):
            members = ", ".join(cluster_groups[cluster_id])
            print(f"  Group {cluster_id}: {members}")

#### Hierarchical Clustering with Spearman Correlation Distance

In [ ]:
from scipy.stats import rankdata

# Define a custom function for Spearman correlation distance
def spearman_correlation_distance(u, v):
    # Calculate ranks for each vector u and v
    rank_u = rankdata(u)
    rank_v = rankdata(v)
    # Calculate Pearson correlation coefficient between the ranks
    corr_matrix = np.corrcoef(rank_u, rank_v)
    corr = corr_matrix[0, 1]

    # Handle NaN correlation (e.g., if one or both vectors are constant)
    if np.isnan(corr):
        # A common practice is to assign a neutral distance of 1.0 (equivalent to correlation 0)
        return 1.0
    return 1 - corr

# Calculate Spearman correlation-based distance using the custom metric
spearman_distances_condensed = pdist(scaled_clustering_data, metric=spearman_correlation_distance)

plt.figure(figsize=(18, 12))

for i, method in enumerate(linkage_methods):
    plt.subplot(3, 1, i + 1)
    linked = linkage(spearman_distances_condensed, method=method)
    dendrogram(linked,
               orientation='top',
               labels=clustering_data.index.tolist(),
               distance_sort='descending',
               show_leaf_counts=True,
               color_threshold=0.7 * max(linked[:,2])) # Adjust threshold for better visibility
    plt.title(f'Hierarchical Clustering (Spearman - {method} Linkage)')
    plt.xlabel('縣市別')
    plt.ylabel('Distance')

plt.tight_layout()
plt.show()

### Scipy vs. Sklearn: 實務操作對照
以下展示如何使用 Scipy 探索結構，以及如何使用 Sklearn 進行快速分群標記。

In [ ]:
import scipy.cluster.hierarchy as sch
from sklearn.cluster import AgglomerativeClustering

# --- 1. Scipy 方式: 重點在於結構與矩陣 ---
# 產出 Z (Linkage Matrix)
Z = sch.linkage(pearson_distances, method='ward')
# 根據 Z 畫圖或切分群組 (例如切成 4 群)
scipy_labels = sch.fcluster(Z, t=4, criterion='maxclust')

# --- 2. Sklearn 方式: 重點在於 Fit/Predict API ---
# 直接定義模型並得到結果
sk_model = AgglomerativeClustering(n_clusters=4, linkage='ward')
sklearn_labels = sk_model.fit_predict(scaled_clustering_data)

# 對照結果 (注意: 標籤編號可能不同，但分群結構應一致)
comparison_df = pd.DataFrame({
    'County': clustering_data_clean.index,
    'Scipy_Group': scipy_labels,
    'Sklearn_Group': sklearn_labels
})

print("Scipy vs Sklearn Label Comparison (k=4):")
display(comparison_df.head())

# 註：Scipy 的 Z 可以直接餵給 sch.dendrogram(Z) 畫圖，而 Sklearn 則需額外計算才能畫圖。

Python `PCA` vs R `prcomp`
```
PYTHON
--- PCA 變異解釋量分析 (排除連江縣) ---
各主成分解釋變異比 (前7個): ['54.12%', '16.04%', '7.63%', '6.79%', '2.79%', '2.31%', '2.18%']
累積解釋變異比 (前7個): ['54.12%', '70.16%', '77.78%', '84.57%', '87.36%', '89.66%', '91.84%']
--------------------------------------

> summary(pca_cov)
Importance of components:
                           PC1     PC2      PC3     PC4     PC5     PC6     PC7     PC8     PC9
Standard deviation     27.0209 14.7113 10.14279 9.56772 6.13232 5.57829 5.42514 4.78211 4.22307
Proportion of Variance  0.5412  0.1604  0.07625 0.06785 0.02787 0.02306 0.02182 0.01695 0.01322
Cumulative Proportion   0.5412  0.7016  0.77783 0.84568 0.87356 0.89662 0.91844 0.93539 0.94860
                          PC10   PC11    PC12    PC13    PC14    PC15    PC16    PC17    PC18    PC19
Standard deviation     3.94293 3.4465 3.05908 2.83332 2.73521 2.35513 1.92191 1.82980 1.36005 1.26844
Proportion of Variance 0.01152 0.0088 0.00694 0.00595 0.00555 0.00411 0.00274 0.00248 0.00137 0.00119
Cumulative Proportion  0.96013 0.9689 0.97587 0.98182 0.98736 0.99147 0.99421 0.99669 0.99806 0.99926
                          PC20      PC21
Standard deviation     1.00085 7.281e-15
Proportion of Variance 0.00074 0.000e+00
Cumulative Proportion  1.00000 1.000e+00

PYTHON
--- PCA 變異解釋量分析 (排除連江縣) ---
各主成分解釋變異比 (前7個): ['53.60%', '13.58%', '8.19%', '6.92%', '3.41%', '2.57%', '2.51%']
累積解釋變異比 (前7個): ['53.60%', '67.18%', '75.37%', '82.29%', '85.71%', '88.28%', '90.79%']
--------------------------------------

> summary(pca_cor)
Importance of components:
                         PC1    PC2     PC3     PC4     PC5     PC6     PC7     PC8     PC9   PC10
Standard deviation     4.856 2.4446 1.89855 1.74551 1.22511 1.06398 1.05137 0.94534 0.84543 0.7206
Proportion of Variance 0.536 0.1358 0.08192 0.06925 0.03411 0.02573 0.02512 0.02031 0.01624 0.0118
Cumulative Proportion  0.536 0.6718 0.75370 0.82295 0.85706 0.88279 0.90791 0.92822 0.94446 0.9563
                          PC11    PC12    PC13    PC14    PC15    PC16    PC17    PC18    PC19    PC20
Standard deviation     0.68956 0.56794 0.53407 0.50837 0.44708 0.34376 0.32179 0.25798 0.24483 0.18573
Proportion of Variance 0.01081 0.00733 0.00648 0.00587 0.00454 0.00269 0.00235 0.00151 0.00136 0.00078
Cumulative Proportion  0.96707 0.97440 0.98089 0.98676 0.99130 0.99399 0.99634 0.99785 0.99922 1.00000
                            PC21
Standard deviation     1.202e-15
Proportion of Variance 0.000e+00
Cumulative Proportion  1.000e+00
```